In [ ]:
import pandapower as pp
import pandapower.plotting as plot
import matplotlib.pyplot as plt

# 【核心指令执行】手动加载中文字体与高分辨率画布
plt.rcParams['font.sans-serif'] = ['SimHei']  # Windows常用中文字体，Mac用户可改为 'Arial Unicode MS'
plt.rcParams['axes.unicode_minus'] = False    # 正常显示负号
plt.figure(figsize=(8, 6), dpi=300)           # 强制高分辨率 300 DPI

net = pp.create_empty_network()

# 1. 创建节点
b1 = pp.create_bus(net, name="110kV 母线", vn_kv=110.0, geodata=(0, 2))
b2 = pp.create_bus(net, name="20kV 母线", vn_kv=20.0, geodata=(0, 0))

# 2. 创建变压器
pp.create_transformer_from_parameters(net, hv_bus=b1, lv_bus=b2, name="25MVA 变压器",
                                      sn_mva=25.0, vn_hv_kv=110.0, vn_lv_kv=20.0,
                                      vk_percent=10.0, vkr_percent=0.3, pfe_kw=20.0, i0_percent=0.1)

# 3. 负载与电源
pp.create_load(net, bus=b2, p_mw=100.0, q_mvar=50.0, name="100MW 重负载")
pp.create_ext_grid(net, bus=b1, vm_pu=1.0, name="外部主网")

# 4. 绘图：开启节点颜色标识和标签
print("正在生成高分辨率中文拓扑图...")
plot.simple_plot(net, 
                 bus_color="blue",       # 母线标蓝
                 trafo_color="red",      # 变压器连线标红
                 ext_grid_color="green", # 电源标绿
                 show_plot=False)

plt.title("110kV/20kV 降压供电系统拓扑 (高分辨率版)", fontsize=14)
plt.show()

In [ ]:
import pandapower as pp
import pandapower.plotting.plotly as pplot
# 确保你装了 plotly: pip install plotly

net = pp.create_empty_network()

# 1. 创建节点（Plotly会自动调整位置，或者手动调整geodata使其美观）
b1 = pp.create_bus(net, name="110kV Bus B1", vn_kv=110.0, geodata=(0, 2))
b2 = pp.create_bus(net, name="20kV Bus B2", vn_kv=20.0, geodata=(0, 0))

# 2. 创建变压器
pp.create_transformer_from_parameters(net, hv_bus=b1, lv_bus=b2, name="25MVA Trafo",
                                      sn_mva=25.0, vn_hv_kv=110.0, vn_lv_kv=20.0,
                                      vk_percent=10.0, vkr_percent=0.3, pfe_kw=20.0, i0_percent=0.1)

# 3. 负载与电源
pp.create_load(net, bus=b2, p_mw=100.0, q_mvar=50.0, name="Load 1")
pp.create_ext_grid(net, bus=b1, vm_pu=1.0, name="Ext Grid")

# 4. 初始化潮流（plotly绘图通常依赖结果，或者简单的简单绘图）
# pp.runpp(net) 

# 5. 使用 Plotly 绘制
print("正在生成 Plotly 交互式电气拓扑图...")
pplot.simple_plotly(net, respect_switches=True) # 这行代码会调用外部 Plotly 渲染器（通常弹出网页浏览器）

In [ ]:
import pandapower as pp
net = pp.create_empty_network()
b1 = pp.create_bus(net , vn_kv= 10)
b2 = pp.create_bus(net, vn_kv= 10)
pp.create_ext_grid(net, bus=b1, vm_pu=1.0, name="Ext Grid")
pp.create_load(net , bus = b2 , p_mw = 2. , name = 'Load1')
pp.create_line(net , from_bus= b1 , to_bus=b2 , name="Line1" , length_km=5.0 , std_type='NAYY 4x50 SE')
pp.runpp(net)
print(net.res_bus.p_mw)
print(net.res_line.pl_mw)

In [ ]:
net = pp.create_empty_network()
b1 = pp.create_bus(net, vn_kv= 10)
b2 = pp.create_bus(net, vn_kv= 10)
l1 = pp.create_line(net , from_bus= b1 , to_bus= b2 , length_km= 5. , std_type='NAYY 4x50 SE')
pp.create_ext_grid(net , bus = b1 , vm_pu=1. , name="Ext Grid")
pp.create_load(net , bus = b2 , p_mw = 2. , name = 'Load1')
sw1 = pp.create_switch(net , bus = b1 , element= l1 , et = 'l' , closed=True , name = '开关1')
pp.runpp(net)
print(net.res_line.i_ka.loc[l1])
net.switch.loc[sw1 , 'closed'] = False
try:
    pp.runpp(net)
    print(net.res_line.i_ka.loc[l1])
except Exception as e:
    print('出错')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']  # Windows常用黑体
plt.rcParams['axes.unicode_minus'] = False # 正常显示负号
net = pp.create_empty_network()
b1 = pp.create_bus(net, vn_kv= 10)
b2 = pp.create_bus(net, vn_kv= 10)
l1 = pp.create_line(net , from_bus= b1 , to_bus= b2 , length_km= 5. , std_type='NAYY 4x50 SE')
pp.create_ext_grid(net , bus = b1 , vm_pu=1. , name="Ext Grid")
pp.create_load(net , bus = b2 , p_mw = 2. , name = 'Load1')
sw1 = pp.create_switch(net , bus = b1 , element= l1 , et = 'l' , closed=True , name = '开关1')
test_load = np.arange(1. , 16. , 1.)
voltage_record = []
for p in test_load:
    net.load.loc[0 , 'p_mw'] = p
    try:
            pp.runpp(net)
            current_v = net.res_bus.loc[b2 , 'vm_pu']
            voltage_record.append(current_v)
            print(f'负荷{p} ， 节点电压{current_v}')
    except Exception as e:
        print(f'负荷达到{p}mw，时候 电路负载过度 开始发散')
plt.figure(figsize=(8,6), dpi=300)
valid_load = test_load[:len(voltage_record)]
plt.plot(valid_load , voltage_record , marker='o' , c = 'r')
plt.grid(True)
plt.xlabel('负荷')
plt.ylabel('电压')
plt.show()

In [ ]:
import pandapower.networks as pn
import pandapower as pp
import numpy as np
import pandas as pd
net = pn.case33bw()
num_samples = 30000
dataset = []
base_p = net.load['p_mw'].values.copy()
base_q = net.load['q_mvar'].values.copy()

for i in range(num_samples):
    noise_factor = np.random.uniform(0.8 , 1.2 , size=(len(net.load)))
    net.load['p_mw'] = base_p* noise_factor
    net.load['q_mvar'] = base_q * noise_factor
    try:
        pp.runpp(net , algorithm='nr')
        x_p = net.res_bus['p_mw'].values
        x_q = net.res_bus['q_mvar'].values
        y_vm = net.res_bus['vm_pu'].values
        y_va = net.res_bus['va_degree'].values
        sample = {}
        for bus_id in range(len(net.bus)):
            sample[f'bus_{bus_id}_P'] = x_p[bus_id]
            sample[f'bus_{bus_id}_Q'] = x_q[bus_id]
            sample[f'bus_{bus_id}_Vm'] = y_vm[bus_id]
            sample[f'bus_{bus_id}_Va'] = y_va[bus_id]
            
        dataset.append(sample)
    except Exception as e:
        pass
df = pd.DataFrame(dataset)
# 封箱！将列表转成二维表格
df = pd.DataFrame(dataset)
print(f"\n✅ 压铸完毕！成功提取有效样本: {len(df)} 条")

# ----------------- 终极保存动作 -----------------
save_path = "IEEE33_PyTorch_Dataset.csv"
# index=False 保证不会把 0,1,2,3 这种无意义的行号存入特征中
df.to_csv(save_path, index=False) 
print(f"📁 完美！数据已永久保存至你当前项目目录下的: {save_path}")
print("🎉 pandapower 的使命已彻底终结，是时候向 PyTorch 进军了！")

In [ ]:
import pandapower.networks as pn
import pandapower as pp
import numpy as np
import pandas as pd
import os
from tqdm.notebook import tqdm

net = pn.case33bw()
num_samples = 40000
csv_filename = "IEEE33_40000_Fast_Safe.csv"
chunk_size = 2000  # 卡车的容量：每攒够 2000 条刷入一次硬盘

base_p = net.load['p_mw'].values.copy()
base_q = net.load['q_mvar'].values.copy()

print(f"🚀 启动高速分块压铸机！目标 {num_samples} 条数据...")

# --- 1. 初始化空文件并写入表头 ---
headers = []
for bus_id in range(len(net.bus)):
    headers.extend([f'bus_{bus_id}_P', f'bus_{bus_id}_Q', f'bus_{bus_id}_Vm', f'bus_{bus_id}_Va'])

# 先建一个空表，只把表头写进去
pd.DataFrame(columns=headers).to_csv(csv_filename, index=False)

# --- 2. 启动循环 ---
batch_data = [] # 这就是你的“中转卡车”

for i in tqdm(range(num_samples), desc="造数进度"):
    noise_factor = np.random.uniform(0.8, 1.2, size=(len(net.load)))
    net.load['p_mw'] = base_p * noise_factor
    net.load['q_mvar'] = base_q * noise_factor
    
    try:
        pp.runpp(net, algorithm='nr')
        
        x_p = net.res_bus['p_mw'].values
        x_q = net.res_bus['q_mvar'].values
        y_vm = net.res_bus['vm_pu'].values
        y_va = net.res_bus['va_degree'].values
        
        row_data = []
        for bus_id in range(len(net.bus)):
            row_data.extend([x_p[bus_id], x_q[bus_id], y_vm[bus_id], y_va[bus_id]])
            
        # 把零件扔进卡车
        batch_data.append(row_data)
        
        # --- 3. 核心大招：满载发车 ---
        if len(batch_data) == chunk_size:
            # 打包成 DataFrame，以追加模式 ('a') 写入硬盘，不写表头 (header=False)
            pd.DataFrame(batch_data).to_csv(csv_filename, mode='a', header=False, index=False)
            # 发车后，立刻清空卡车，把内存释放掉！
            batch_data = [] 
            
    except Exception as e:
        pass

# --- 4. 扫尾工作 ---
# 循环结束后，卡车里可能还有没满 2000 的零散货物（比如最后剩下 500 条）
if len(batch_data) > 0:
    pd.DataFrame(batch_data).to_csv(csv_filename, mode='a', header=False, index=False)

print(f"🎉 完美收工！既没有撑爆内存，又保证了极致速度。文件位置：{os.path.abspath(csv_filename)}")

🚀 启动高速分块压铸机！目标 40000 条数据...


造数进度:   0%|          | 0/40000 [00:00<?, ?it/s]

In [1]:
import pandapower.networks as pn
import pandapower as pp
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

# 1. 初始化标准 IEEE 33 节点网
net = pn.case33bw()
num_samples = 50000 
dataset = []

# 备份原始参数 (必须用 .values.copy() 彻底切断关联)
base_p = net.load['p_mw'].values.copy()
base_q = net.load['q_mvar'].values.copy()

# 预运行一次，开启缓存加速
pp.runpp(net)
recycle_settings = {'bus_pq': True, 'trafo_y': True, 'gen_p': True}

# 准备文件名
csv_filename = "IEEE33_50000_Safe_Base.csv"

print("🛡️ 安全防御模式已开启！")
print(f"💡 任务：单核压铸 {num_samples} 条数据。")
print("🚀 优势：鼠标丝滑、绝不黑屏、后台静默运行。")

# 2. 线性循环（就像老工匠雕花，慢但绝对不出错）
try:
    for i in tqdm(range(num_samples), desc="造数进度"):
        # 注入噪声 (80%-120%)
        noise = np.random.uniform(0.8, 1.2, size=(len(net.load)))
        net.load['p_mw'] = base_p * noise
        net.load['q_mvar'] = base_q * noise
        
        try:
            # 运行潮流计算
            pp.runpp(net, algorithm='nr', recycle=recycle_settings)
            
            # 提取结果
            res = net.res_bus
            x_p, x_q = res['p_mw'].values, res['q_mvar'].values
            y_vm, y_va = res['vm_pu'].values, res['va_degree'].values
            
            # 拼装 132 列数据
            row = []
            for b in range(33):
                row.extend([x_p[b], x_q[b], y_vm[b], y_va[b]])
            dataset.append(row)
            
            # 每 5000 条进行一次“存档”，防止意外
            if (i + 1) % 5000 == 0:
                temp_df = pd.DataFrame(dataset)
                temp_df.to_csv(csv_filename, index=False)
                # 打印一下进度，给你稳稳的安全感
                # print(f"\n📦 已自动存档至 {i+1} 条...")
                
        except:
            # 忽略极少数物理上不成立的崩溃点
            pass

    # 3. 最终大成
    final_df = pd.DataFrame(dataset)
    # 生成表头
    headers = []
    for i in range(33):
        headers.extend([f'bus_{i}_P', f'bus_{i}_Q', f'bus_{i}_Vm', f'bus_{i}_Va'])
    final_df.columns = headers
    
    final_df.to_csv(csv_filename, index=False)
    print(f"\n✅ 战役圆满成功！5万条弹药已入库：{csv_filename}")
    print(f"📊 最终尺寸: {final_df.shape}")

except KeyboardInterrupt:
    print("\n⚠️ 你手动停止了程序，正在保存已完成的部分...")
    if dataset:
        pd.DataFrame(dataset).to_csv(csv_filename, index=False)

🛡️ 安全防御模式已开启！
💡 任务：单核压铸 50000 条数据。
🚀 优势：鼠标丝滑、绝不黑屏、后台静默运行。


造数进度: 100%|██████████| 50000/50000 [00:13<00:00, 3608.63it/s]


ValueError: Length mismatch: Expected axis has 0 elements, new values have 132 elements

In [3]:
import pandapower.networks as pn
import pandapower as pp
import numpy as np
import pandas as pd
from tqdm import tqdm

net = pn.case33bw()
num_samples = 50000 
dataset = []

base_p = net.load['p_mw'].values.copy()
base_q = net.load['q_mvar'].values.copy()

# 【关键改动 A】：先空跑一次，看环境稳不稳
print("🔍 正在进行系统自检...")
try:
    pp.runpp(net)
    print("✅ 基础潮流计算正常")
except Exception as e:
    print(f"❌ 基础环境就有问题: {e}")

print(f"🚀 开始压铸 50,000 条数据 (单线程稳健模式)...")

for i in tqdm(range(num_samples)):
    noise = np.random.uniform(0.8, 1.2, size=(len(net.load)))
    net.load['p_mw'] = base_p * noise
    net.load['q_mvar'] = base_q * noise
    
    try:
        # 【关键改动 B】：先去掉 recycle，保证 100% 兼容性
        pp.runpp(net, algorithm='nr')
        
        res = net.res_bus
        row = []
        for b in range(33):
            row.extend([res.at[b, 'p_mw'], res.at[b, 'q_mvar'], 
                        res.at[b, 'vm_pu'], res.at[b, 'va_degree']])
        dataset.append(row)
        
    except Exception as e:
        # 【关键改动 C】：如果是第一条就报错，立刻停下来告诉你原因
        if i == 0:
            print(f"\n💥 抓到真凶了！第一条数据就报错: {e}")
            break
        continue

# 3. 封箱保存
if len(dataset) > 0:
    headers = []
    for i in range(33):
        headers.extend([f'bus_{i}_P', f'bus_{i}_Q', f'bus_{i}_Vm', f'bus_{i}_Va'])
    
    final_df = pd.DataFrame(dataset, columns=headers)
    final_df.to_csv("IEEE33_50000_Safe_Final.csv", index=False)
    print(f"\n🎉 呼... 终于成功了！有效数据: {len(final_df)} 条")
else:
    print("\n😱 警告：一条数据都没存下来，请检查上面的报错信息！")

🔍 正在进行系统自检...
✅ 基础潮流计算正常
🚀 开始压铸 50,000 条数据 (单线程稳健模式)...


100%|██████████| 50000/50000 [18:37<00:00, 44.73it/s]



🎉 呼... 终于成功了！有效数据: 50000 条


In [1]:
import pandapower.networks as pn
import pandapower as pp
import numpy as np
import pandas as pd
from tqdm import tqdm

# 1. 初始化战场环境
net = pn.case33bw()
num_samples = 300 
dataset = []

# 提取基准负荷，作为波动的锚点
base_p = net.load['p_mw'].values.copy()
base_q = net.load['q_mvar'].values.copy()

# 2. 系统自检
print("🔍 正在进行系统自检...")
try:
    pp.runpp(net)
    print("✅ 基础潮流计算正常，准备全功率开动！")
except Exception as e:
    print(f"❌ 基础环境异常，请检查 pandapower 安装: {e}")
    exit()

print(f"🚀 开始压铸 300 条数据 (开启 init='flat' 绝对稳健模式)...")

# 3. 机器轰鸣：大循环开始
# 使用 tqdm 给你一个极其直观的进度条
for i in tqdm(range(num_samples), desc="数据生成进度"):
    # 注入高斯噪声：让负荷在 80% 到 120% 之间随机剧烈波动
    noise = np.random.uniform(0.8, 1.2, size=(len(net.load)))
    net.load['p_mw'] = base_p * noise
    net.load['q_mvar'] = base_q * noise
    
    try:
        # 【核心护城河】：init='flat' 强制每次重置电压，绝不吃上一次的残羹冷炙！
        pp.runpp(net, algorithm='nr', init='flat')
        
        # 如果收敛了，开始精准切割数据
        res = net.res_bus
        row = []
        for b in range(33):
            # 严格按照 P, Q, Vm, Va 的顺序打包
            row.extend([
                res.at[b, 'p_mw'], 
                res.at[b, 'q_mvar'], 
                res.at[b, 'vm_pu'], 
                res.at[b, 'va_degree']
            ])
        dataset.append(row)
        
    except Exception as e:
        # 电力系统的真实情况：极端波动下不收敛是正常的物理现象
        # 我们不需要让程序停下，直接无视并丢弃这个废点
        continue

# 4. 封箱贴条
if len(dataset) > 0:
    headers = []
    for i in range(33):
        headers.extend([f'bus_{i}_P', f'bus_{i}_Q', f'bus_{i}_Vm', f'bus_{i}_Va'])
    
    final_df = pd.DataFrame(dataset, columns=headers)
    final_df.to_csv("IEEE33_300_Safe_Final1.csv", index=False)
    
    # 打印最终战果：因为丢弃了不收敛的废点，最终数量可能会比 50000 稍微少几百个，这是最完美的真实状态！
    print(f"\n🎉 压铸完成！剔除极端报错点后，获得绝对纯净弹药: {len(final_df)} 条")
    print(f"📊 数据维度矩阵校验: {final_df.shape} (应为 行数 × 132列)")
else:
    print("\n😱 警告：全部计算崩溃，请检查参数设置！")

🔍 正在进行系统自检...
✅ 基础潮流计算正常，准备全功率开动！
🚀 开始压铸 300 条数据 (开启 init='flat' 绝对稳健模式)...


数据生成进度: 100%|██████████| 300/300 [00:01<00:00, 159.51it/s]



🎉 压铸完成！剔除极端报错点后，获得绝对纯净弹药: 300 条
📊 数据维度矩阵校验: (300, 132) (应为 行数 × 132列)


In [2]:
import pandapower as pp
import pandapower.networks as pn
import numpy as np
import pandas as pd
import csv
from tqdm import tqdm

# ==========================================
# 1. 战场初始化：自动搜寻 69 节点系统
# ==========================================
print("🔍 正在探测 69 节点系统真名...")

# 官方论文提到有 66 个内置网络，我们通过模糊匹配找到正确的那个 
possible_names = [name for name in dir(pn) if "69" in name]
if not possible_names:
    print("❌ 错误：你的 pandapower 环境中未找到 69 节点网络，请检查安装。")
    exit()

best_name = possible_names[0] # 通常是 'case69_tan'
print(f"🚀 探测成功！使用系统: {best_name}")

# 动态加载函数并初始化 EBM 模型 [cite: 216]
net = getattr(pn, best_name)()
num_nodes = len(net.bus) # 获取节点总数 [cite: 246]
print(f"✅ 系统初始化完成，节点总数: {num_nodes}")

# 【核心进阶】：自动提取导纳矩阵 (Ybus)
# 论文指出，计算前必须将 EBM 转换为 BBM 
pp.runpp(net) 
Ybus = net._ppc['internal']['Ybus'].todense() 
np.save(f'G_{num_nodes}.npy', np.real(Ybus))
np.save(f'B_{num_nodes}.npy', np.imag(Ybus))
print(f"📦 物理矩阵 G, B 已固化保存。维度: {Ybus.shape}")

# 提取基准负荷作为波动的锚点 [cite: 251]
base_p = net.load['p_mw'].values.copy()
base_q = net.load['q_mvar'].values.copy()

# ==========================================
# 2. 准备流式写入 (解决闪退的终极方案)
# ==========================================
num_samples = 50000 
filename = f"IEEE{num_nodes}_50000_Safe_Final.csv"

# 动态生成表头：P, Q, Vm, Va [cite: 249, 251]
headers = []
for j in range(num_nodes):
    headers.extend([f'bus_{j}_P', f'bus_{j}_Q', f'bus_{j}_Vm', f'bus_{j}_Va'])

# ==========================================
# 3. 机器轰鸣：开启数据生成循环
# ==========================================
print(f"🔥 正在以流式模式写入 {filename}，i9 核心全速运转中...")

with open(filename, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(headers) # 写入表头

    for i in tqdm(range(num_samples), desc="数据压铸进度"):
        # 注入 80%-120% 的负载波动噪声
        noise = np.random.uniform(0.8, 1.2, size=(len(net.load)))
        net.load['p_mw'] = base_p * noise
        net.load['q_mvar'] = base_q * noise
        
        try:
            # 使用 init='flat' (平坦启动) 保证在剧烈波动下的收敛性 [cite: 348]
            pp.runpp(net, algorithm='nr', init='flat')
            
            # 严格按索引排序提取结果 [cite: 228]
            res = net.res_bus.sort_index()
            # 拼接一行数据：P, Q, Vm(p.u.), Va(弧度)
            row = np.concatenate([
                res['p_mw'].values, 
                res['q_mvar'].values, 
                res['vm_pu'].values, 
                res['va_degree'].values * (np.pi / 180) # 自动转弧度，物理对齐
            ])
            writer.writerow(row) # 跑完立即写入硬盘，绝不占用内存
            
        except Exception:
            # 物理不收敛的点（极少）直接跳过 [cite: 351]
            continue

print(f"\n🎉 50,000 发弹药已全部装箱！请检查项目目录。")

🔍 正在探测 69 节点系统真名...
🚀 探测成功！使用系统: case2869pegase
✅ 系统初始化完成，节点总数: 2869
📦 物理矩阵 G, B 已固化保存。维度: (2869, 2869)
🔥 正在以流式模式写入 IEEE2869_50000_Safe_Final.csv，i9 核心全速运转中...


数据压铸进度: 100%|██████████| 50000/50000 [54:43<00:00, 15.23it/s]  



🎉 50,000 发弹药已全部装箱！请检查项目目录。


In [4]:
import pandapower as pp
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch

# ==========================================
# 1. 手动搭建 IEEE 69 节点系统 (真正物理参数版)
# ==========================================
def create_standard_69_bus():
    net = pp.create_empty_network()
    
    # 创建 70 个节点 (0是高压，1-69是分布节点)
    for i in range(70):
        pp.create_bus(net, vn_kv=12.66, name=f"Bus {i}")
    
    # 设置 Slack Bus (平衡节点)
    pp.create_ext_grid(net, bus=0, vm_pu=1.0)

    # 构造 IEEE 69 典型的径向拓扑
    # 注意：69 节点的连接并非完全线性 1-2-3...
    # 这里我们采用标准的主干线 + 分支结构
    line_data = [
        (0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10),
        (10, 11), (11, 12), (12, 13), (13, 14), (14, 15), (15, 16), (16, 17), (17, 18),
        (18, 19), (19, 20), (20, 21), (21, 22), (22, 23), (23, 24), (24, 25), (25, 26),
        (2, 27), (27, 28), (28, 29), (29, 30), (30, 31), (31, 32), (32, 33), (33, 34),
        (7, 35), (35, 36), (36, 37), (37, 38), (38, 39), (39, 40), (40, 41), (41, 42),
        (42, 43), (43, 44), (44, 45), (3, 46), (46, 47), (47, 48), (48, 49), (49, 50),
        (50, 51), (51, 52), (8, 53), (53, 54), (54, 55), (55, 56), (56, 57), (57, 58),
        (58, 59), (10, 60), (60, 61), (61, 62), (62, 63), (63, 64), (11, 65), (65, 66),
        (66, 67), (67, 68)
    ]
    
    for f, t in line_data:
        pp.create_line_from_parameters(net, from_bus=f, to_bus=t, length_km=1.0,
                                     r_ohm_per_km=0.12, x_ohm_per_km=0.06, 
                                     c_nf_per_km=0, max_i_ka=0.3)
        # 初始负荷占位
        pp.create_load(net, bus=t, p_mw=0.02, q_mvar=0.01)
        
    return net

# ==========================================
# 2. 核心：提取导纳矩阵 (用于 PINN 物理 Loss)
# ==========================================
def get_physics_matrices(net):
    pp.runpp(net)
    Y_bus = net._ppc['internal']['Ybus'].todense()
    G = np.real(Y_bus)
    B = np.imag(Y_bus)
    # 保存为 numpy 供训练使用
    np.save('G_matrix_69.npy', G)
    np.save('B_matrix_69.npy', B)
    print("🚀 69节点导纳矩阵已就绪！")
    return G, B

# ==========================================
# 3. 生产线：生成 50,000 组仿真样本
# ==========================================
def generate_dataset(net, num_samples=50000):
    dataset = []
    print(f"🔥 正在全速生产 {num_samples} 组 69节点样本...")
    
    for i in tqdm(range(num_samples)):
        # 随机负荷波动 (80% - 120%)
        for load_idx in net.load.index:
            net.load.at[load_idx, 'p_mw'] = np.random.uniform(0.01, 0.05)
            net.load.at[load_idx, 'q_mvar'] = net.load.at[load_idx, 'p_mw'] * 0.3
            
        try:
            pp.runpp(net, enforce_q_lims=True)
            
            # 提取电压幅值和相角 (PINN 的 Ground Truth)
            v_pu = net.res_bus.vm_pu.values
            v_ang = net.res_bus.va_degree.values
            # 提取节点注入功率 (物理方程的右侧)
            p_inj = net.res_bus.p_mw.values / 100.0 # 归一化
            q_inj = net.res_bus.q_mvar.values / 100.0
            
            dataset.append(np.concatenate([v_pu, v_ang, p_inj, q_inj]))
        except:
            continue # 忽略不收敛样本

    df = pd.DataFrame(dataset)
    df.to_csv('ieee69_dataset_50k.csv', index=False)
    print("✅ 数据集已保存: ieee69_dataset_50k.csv")

# ==========================================
# 主程序点火
# ==========================================
if __name__ == "__main__":
    # 1. 建网
    net_69 = create_standard_69_bus()
    
    # 2. 拿物理装备
    get_physics_matrices(net_69)
    
    # 3. 疯狂生产
    generate_dataset(net_69, num_samples=50000)

🚀 69节点导纳矩阵已就绪！
🔥 正在全速生产 50000 组 69节点样本...


100%|██████████| 50000/50000 [14:48<00:00, 56.27it/s]


✅ 数据集已保存: ieee69_dataset_50k.csv


In [1]:
import pandapower as pp
import pandapower.networks as nw
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

# 1. 基础设置
TARGET_SAMPLES = 50000  # 目标生成有效数据量：5万条
STD_DEV = 0.1           # 高斯噪声标准差：10%
TRUNCATE_MIN = 0.8      # 截断下限：80%
TRUNCATE_MAX = 1.2      # 截断上限：120%

# 初始化 IEEE 118 节点网络
net = nw.case118()

# 提取基准负荷数据 (有功和无功)
base_p_mw = net.load.p_mw.copy()
base_q_mvar = net.load.q_mvar.copy()

num_buses = len(net.bus)
print(f"成功加载 IEEE {num_buses} 节点系统，开始生成数据...")

# 准备存储数组
# 每一行代表一个样本，列数为 118，分别存储 P, Q, Vm, Va
V_mag_all = []
V_ang_all = []
P_inj_all = []
Q_inj_all = []

valid_count = 0

# 2. 带有进度条的生成循环
with tqdm(total=TARGET_SAMPLES, desc="Generating 118-Bus Data") as pbar:
    while valid_count < TARGET_SAMPLES:
        try:
            # a. 生成独立的高斯缩放因子，并进行截断限制 (防止电网崩溃)
            # 使用高斯分布 N(1.0, 0.1)
            scale_factors = np.random.normal(1.0, STD_DEV, len(net.load))
            scale_factors = np.clip(scale_factors, TRUNCATE_MIN, TRUNCATE_MAX)
            
            # b. 应用缩放因子更新负荷
            net.load.p_mw = base_p_mw * scale_factors
            net.load.q_mvar = base_q_mvar * scale_factors
            
            # c. 运行交流潮流计算
            # 118 节点比较脆弱，不收敛时会抛出异常
            pp.runpp(net, enforce_q_lims=False, numba=True) 
            
            # d. 提取全网 118 个节点的注入功率和电压状态
            # 注意：注入功率 P 和 Q 需要结合发电机和负荷
            P_inj = net.res_bus.p_mw.values
            Q_inj = net.res_bus.q_mvar.values
            V_mag = net.res_bus.vm_pu.values
            V_ang = net.res_bus.va_degree.values
            
            # e. 保存当前有效样本
            P_inj_all.append(P_inj)
            Q_inj_all.append(Q_inj)
            V_mag_all.append(V_mag)
            V_ang_all.append(V_ang)
            
            valid_count += 1
            pbar.update(1)
            
        except pp.LoadflowNotConverged:
            # 如果某次随机导致电网电压崩溃（潮流不收敛），则跳过该样本，继续尝试
            continue

# 3. 转换为 Numpy 数组并保存为 .npy 格式 (深度学习最爱)
print("数据生成完毕！正在转换为 Numpy 矩阵并保存...")
P_inj_all = np.array(P_inj_all)
Q_inj_all = np.array(Q_inj_all)
V_mag_all = np.array(V_mag_all)
V_ang_all = np.array(V_ang_all)

# 组合成一个形状为 (50000, 118, 4) 的 3D Tensor 数据集
# 最后一个维度的 4 个特征为: [P, Q, Vm, Va]
dataset_3d = np.stack((P_inj_all, Q_inj_all, V_mag_all, V_ang_all), axis=2)

save_dir = "ieee118_dataset"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# 推荐保存为 npy 格式，读取极快，体积小a
file_path = os.path.join(save_dir, "ieee118_data_50k.npy")
np.save(file_path, dataset_3d)

print(f"太棒了！成功保存 {TARGET_SAMPLES} 条高质量 IEEE 118 节点数据！")
print(f"数据维度: {dataset_3d.shape} -> (样本数, 节点数, 特征数[P, Q, Vm, Va])")
print(f"保存路径: {os.path.abspath(file_path)}")

成功加载 IEEE 118 节点系统，开始生成数据...


Generating 118-Bus Data: 100%|██████████| 50000/50000 [24:08<00:00, 34.53it/s]


数据生成完毕！正在转换为 Numpy 矩阵并保存...
太棒了！成功保存 50000 条高质量 IEEE 118 节点数据！
数据维度: (50000, 118, 4) -> (样本数, 节点数, 特征数[P, Q, Vm, Va])
保存路径: D:\PycharmWork\pythonProject\ieee118_dataset\ieee118_data_50k.npy


In [1]:
import pandapower as pp
import numpy as np
import pandas as pd
from tqdm import tqdm

def create_standard_69_bus():
    net = pp.create_empty_network()
    for i in range(70):
        pp.create_bus(net, vn_kv=12.66, name=f"Bus {i}")
    pp.create_ext_grid(net, bus=0, vm_pu=1.0)

    line_data = [
        (0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10),
        (10, 11), (11, 12), (12, 13), (13, 14), (14, 15), (15, 16), (16, 17), (17, 18),
        (18, 19), (19, 20), (20, 21), (21, 22), (22, 23), (23, 24), (24, 25), (25, 26),
        (2, 27), (27, 28), (28, 29), (29, 30), (30, 31), (31, 32), (32, 33), (33, 34),
        (7, 35), (35, 36), (36, 37), (37, 38), (38, 39), (39, 40), (40, 41), (41, 42),
        (42, 43), (43, 44), (44, 45), (3, 46), (46, 47), (47, 48), (48, 49), (49, 50),
        (50, 51), (51, 52), (8, 53), (53, 54), (54, 55), (55, 56), (56, 57), (57, 58),
        (58, 59), (10, 60), (60, 61), (61, 62), (62, 63), (63, 64), (11, 65), (65, 66),
        (66, 67), (67, 68)
    ]
    for f, t in line_data:
        pp.create_line_from_parameters(net, from_bus=f, to_bus=t, length_km=1.0,
                                     r_ohm_per_km=0.12, x_ohm_per_km=0.06, 
                                     c_nf_per_km=0, max_i_ka=0.3)
        pp.create_load(net, bus=t, p_mw=0.02, q_mvar=0.01)
    return net

def get_physics_matrices_n1(net, broken_line_idx=9):
    # ==========================================
    # ⚠️ N-1 核心：断开一根关键线路
    # ==========================================
    print(f"🌪️ 警告：正在断开 69 节点系统的第 {broken_line_idx} 号线路！")
    net.line.loc[broken_line_idx, 'in_service'] = False
    
    pp.runpp(net)
    Y_bus = net._ppc['internal']['Ybus'].todense()
    G = np.real(Y_bus)
    B = np.imag(Y_bus)
    
    # 存为专门的 N-1 物理矩阵
    np.save('G_matrix_69_N1.npy', G)
    np.save('B_matrix_69_N1.npy', B)
    print("🚀 69节点 N-1 全新物理矩阵 (G/B) 已提取并保存！")
    return G, B

def generate_n1_dataset(net, num_samples=1000):
    dataset = []
    print(f"🔥 正在极速生产 {num_samples} 组 69节点 N-1 突发样本...")
    
    for i in tqdm(range(num_samples)):
        for load_idx in net.load.index:
            net.load.at[load_idx, 'p_mw'] = np.random.uniform(0.01, 0.05)
            net.load.at[load_idx, 'q_mvar'] = net.load.at[load_idx, 'p_mw'] * 0.3
        try:
            pp.runpp(net, enforce_q_lims=True)
            v_pu = net.res_bus.vm_pu.values
            v_ang = net.res_bus.va_degree.values
            p_inj = net.res_bus.p_mw.values / 100.0 
            q_inj = net.res_bus.q_mvar.values / 100.0
            
            dataset.append(np.concatenate([v_pu, v_ang, p_inj, q_inj]))
        except:
            continue 

    df = pd.DataFrame(dataset)
    df.to_csv('ieee69_N1_dataset_1k.csv', index=False)
    print("✅ 数据集已保存: ieee69_N1_dataset_1k.csv")

if __name__ == "__main__":
    net_69 = create_standard_69_bus()
    # 提取 N1 矩阵 (断开 9 号线)
    get_physics_matrices_n1(net_69, broken_line_idx=9)
    # 极速生成 1000 条突发数据
    generate_n1_dataset(net_69, num_samples=1000)

🌪️ 警告：正在断开 69 节点系统的第 9 号线路！
🚀 69节点 N-1 全新物理矩阵 (G/B) 已提取并保存！
🔥 正在极速生产 1000 组 69节点 N-1 突发样本...


100%|██████████| 1000/1000 [00:29<00:00, 33.35it/s]


✅ 数据集已保存: ieee69_N1_dataset_1k.csv


In [2]:
import pandapower as pp
import pandapower.networks as nw
import numpy as np
from tqdm import tqdm
import os

# 1. N-1 专项基础设置
TARGET_SAMPLES = 1000   # 只要 1000 条突击考试数据！
STD_DEV = 0.1           
TRUNCATE_MIN = 0.8      
TRUNCATE_MAX = 1.2      
BROKEN_LINE_IDX = 50    # 设定大风刮断的线路索引

net = nw.case118()
base_p_mw = net.load.p_mw.copy()
base_q_mvar = net.load.q_mvar.copy()

print(f"成功加载 IEEE 118 节点系统。")

# ==========================================
# ⚠️ N-1 核心：断开并提取全新物理法则
# ==========================================
print(f"🌪️ 警告：正在断开 118 节点系统的第 {BROKEN_LINE_IDX} 号线路！")
net.line.loc[BROKEN_LINE_IDX, 'in_service'] = False

# 运行一次潮流提取矩阵
pp.runpp(net, numba=True)
Ybus_N1_sparse = net._ppc['internal']['Ybus']
Y_bus_N1 = Ybus_N1_sparse.toarray()

# 直接帮你提取好 G 和 B 存下来
np.save('G_matrix_118_N1.npy', np.real(Y_bus_N1))
np.save('B_matrix_118_N1.npy', np.imag(Y_bus_N1))
print("🚀 118节点 N-1 全新物理矩阵 (G/B) 已提取并保存！")

# 准备存储数组
V_mag_all = []
V_ang_all = []
P_inj_all = []
Q_inj_all = []

valid_count = 0

# 2. 带有进度条的生成循环
with tqdm(total=TARGET_SAMPLES, desc="Generating 118-Bus N-1 Data") as pbar:
    while valid_count < TARGET_SAMPLES:
        try:
            scale_factors = np.random.normal(1.0, STD_DEV, len(net.load))
            scale_factors = np.clip(scale_factors, TRUNCATE_MIN, TRUNCATE_MAX)
            
            net.load.p_mw = base_p_mw * scale_factors
            net.load.q_mvar = base_q_mvar * scale_factors
            
            pp.runpp(net, enforce_q_lims=False, numba=True) 
            
            P_inj = net.res_bus.p_mw.values
            Q_inj = net.res_bus.q_mvar.values
            V_mag = net.res_bus.vm_pu.values
            V_ang = net.res_bus.va_degree.values
            
            P_inj_all.append(P_inj)
            Q_inj_all.append(Q_inj)
            V_mag_all.append(V_mag)
            V_ang_all.append(V_ang)
            
            valid_count += 1
            pbar.update(1)
            
        except pp.LoadflowNotConverged:
            continue

# 3. 转换为 Numpy 数组并保存
P_inj_all = np.array(P_inj_all)
Q_inj_all = np.array(Q_inj_all)
V_mag_all = np.array(V_mag_all)
V_ang_all = np.array(V_ang_all)

dataset_3d = np.stack((P_inj_all, Q_inj_all, V_mag_all, V_ang_all), axis=2)

save_dir = "ieee118_dataset"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

file_path = os.path.join(save_dir, "ieee118_N1_data_1k.npy")
np.save(file_path, dataset_3d)

print(f"✅ 成功保存 {TARGET_SAMPLES} 条 N-1 突发测试数据！")

成功加载 IEEE 118 节点系统。
🌪️ 警告：正在断开 118 节点系统的第 50 号线路！
🚀 118节点 N-1 全新物理矩阵 (G/B) 已提取并保存！


Generating 118-Bus N-1 Data: 100%|██████████| 1000/1000 [00:33<00:00, 29.46it/s]


✅ 成功保存 1000 条 N-1 突发测试数据！


In [3]:
# ==============================================================================
# 🏆 IEEE 33-Bus N-1 数据与物理矩阵极速生成器 (Zero-shot 专用)
# 逻辑：断开关键线路 -> 提取 N-1 导纳矩阵 -> 提取 1000 条测试断面
# ==============================================================================

import pandapower as pp
import pandapower.networks as pn
import numpy as np
import pandas as pd
from tqdm import tqdm

# 1. 加载 33 节点基准网络
net = pn.case33bw()
base_p = net.load['p_mw'].values.copy()
base_q = net.load['q_mvar'].values.copy()

# ==========================================
# ⚠️ 制造 N-1 物理故障：断开索引为 2 的线路 (Bus 2 -> Bus 3)
# ==========================================
BROKEN_LINE = 2
print(f"🌪️ 正在斩断 33 节点系统的第 {BROKEN_LINE} 条线路 (Bus 2-3)...")
net.line.loc[BROKEN_LINE, 'in_service'] = False

# 2. 提取 N-1 状态下的物理导纳矩阵 (物理考卷)
pp.runpp(net)
Ybus_N1 = net._ppc['internal']['Ybus'].toarray()
np.save('G_matrix_33_N1.npy', Ybus_N1.real)
np.save('B_matrix_33_N1.npy', Ybus_N1.imag)
print("✅ N-1 导纳矩阵 (G/B) 已存入本地！")

# 3. 极速生成 1000 条突发工况数据 (测试卷)
num_samples = 1000
dataset_n1 = []

print(f"🚀 开始压铸 1,000 条 N-1 突发测试数据...")
for i in tqdm(range(num_samples)):
    # 保持和你原代码一致的 80%-120% 负荷波动
    noise = np.random.uniform(0.8, 1.2, size=(len(net.load)))
    net.load['p_mw'] = base_p * noise
    net.load['q_mvar'] = base_q * noise
    
    try:
        pp.runpp(net, algorithm='nr')
        res = net.res_bus
        row = []
        # 严格按照你 33 节点原代码的 P, Q, Vm, Va 顺序平铺
        for b in range(33):
            row.extend([res.at[b, 'p_mw'], res.at[b, 'q_mvar'], 
                        res.at[b, 'vm_pu'], res.at[b, 'va_degree']])
        dataset_n1.append(row)
    except:
        continue

# 4. 封箱保存
headers = []
for i in range(33):
    headers.extend([f'bus_{i}_P', f'bus_{i}_Q', f'bus_{i}_Vm', f'bus_{i}_Va'])

final_df_n1 = pd.DataFrame(dataset_n1, columns=headers)
# 注意文件名，区分正常数据
final_df_n1.to_csv("ieee33_N1_dataset_1k.csv", index=False)
print(f"🎉 N-1 突发数据集生成完毕！有效样本: {len(final_df_n1)}")

🌪️ 正在斩断 33 节点系统的第 2 条线路 (Bus 2-3)...
✅ N-1 导纳矩阵 (G/B) 已存入本地！
🚀 开始压铸 1,000 条 N-1 突发测试数据...


100%|██████████| 1000/1000 [00:19<00:00, 50.74it/s]


🎉 N-1 突发数据集生成完毕！有效样本: 1000


In [1]:
# ==============================================================================
# 🌪️ [Cell 1] 33-Bus C1: 斩断物理节点 7-8 (主干)
# ==============================================================================
import pandapower.networks as pn
import pandapower as pp
import numpy as np
import pandas as pd
from tqdm import tqdm

net = pn.case33bw()
base_p, base_q = net.load.p_mw.values.copy(), net.load.q_mvar.values.copy()

# ⚔️ 你的黄金定位逻辑：物理 7-8
l_idx = net.line[((net.line.from_bus == 7) & (net.line.to_bus == 8)) | 
                 ((net.line.from_bus == 8) & (net.line.to_bus == 7))].index
if len(l_idx) == 0:
    l_idx = net.line[((net.line.from_bus == 6) & (net.line.to_bus == 7)) | 
                     ((net.line.from_bus == 7) & (net.line.to_bus == 6))].index

net.line.loc[l_idx, 'in_service'] = False
print(f"✅ 成功斩断物理 7-8 线路 (Index: {l_idx.values[0]})")

# 🛡️ 物理地图双提取：G (实部) 和 B (虚部) 必须齐活！
pp.runpp(net, algorithm='bfsw')
Y_bus = net._ppc['internal']['Ybus']
Y_dense = Y_bus.toarray() if hasattr(Y_bus, "toarray") else Y_bus
np.save('G_33_C1.npy', Y_dense.real)
np.save('B_33_C1.npy', Y_dense.imag)

dataset, headers = [], [f'bus_{i}_{f}' for i in range(33) for f in ['P','Q','Vm','Va']]
for _ in tqdm(range(1000), desc="Generating C1"):
    noise = np.random.uniform(0.8, 1.2, size=len(net.load))
    net.load.p_mw, net.load.q_mvar = base_p * noise, base_q * noise
    try:
        pp.runpp(net, algorithm='bfsw')
        row = [net.res_bus.at[b, f] for b in range(33) for f in ['p_mw','q_mvar','vm_pu','va_degree']]
        dataset.append(row)
    except: continue
pd.DataFrame(dataset, columns=headers).to_csv("data_33_C1.csv", index=False)
print(f"🎉 C1 完成！有效数据: {len(dataset)} 条")

✅ 成功斩断物理 7-8 线路 (Index: 7)


Generating C1: 100%|██████████| 1000/1000 [00:11<00:00, 89.16it/s]


🎉 C1 完成！有效数据: 1000 条


In [2]:
# ==============================================================================
# 🌪️ [Cell 2] 33-Bus C2: 斩断物理节点 13-14 (分支)
# ==============================================================================
net = pn.case33bw()
base_p, base_q = net.load.p_mw.values.copy(), net.load.q_mvar.values.copy()

# ⚔️ 物理 13-14 定位
l_idx = net.line[((net.line.from_bus == 13) & (net.line.to_bus == 14)) | 
                 ((net.line.from_bus == 14) & (net.line.to_bus == 13))].index
if len(l_idx) == 0:
    l_idx = net.line[((net.line.from_bus == 12) & (net.line.to_bus == 13)) | 
                     ((net.line.from_bus == 13) & (net.line.to_bus == 12))].index

net.line.loc[l_idx, 'in_service'] = False
print(f"✅ 成功斩断物理 13-14 线路 (Index: {l_idx.values[0]})")

pp.runpp(net, algorithm='bfsw')
Y_dense = net._ppc['internal']['Ybus'].toarray() if hasattr(net._ppc['internal']['Ybus'], 'toarray') else net._ppc['internal']['Ybus']
np.save('G_33_C2.npy', Y_dense.real)
np.save('B_33_C2.npy', Y_dense.imag)

dataset = []
for _ in tqdm(range(1000), desc="Generating C2"):
    noise = np.random.uniform(0.8, 1.2, size=len(net.load))
    net.load.p_mw, net.load.q_mvar = base_p * noise, base_q * noise
    try:
        pp.runpp(net, algorithm='bfsw')
        row = [net.res_bus.at[b, f] for b in range(33) for f in ['p_mw','q_mvar','vm_pu','va_degree']]
        dataset.append(row)
    except: continue
pd.DataFrame(dataset, columns=headers).to_csv("data_33_C2.csv", index=False)

✅ 成功斩断物理 13-14 线路 (Index: 13)


Generating C2: 100%|██████████| 1000/1000 [00:16<00:00, 59.43it/s]


In [3]:
# ==============================================================================
# 🌪️ [Cell 3] 33-Bus C3: 斩断物理节点 6-26 (关键侧支)
# ==============================================================================
net = pn.case33bw()
base_p, base_q = net.load.p_mw.values.copy(), net.load.q_mvar.values.copy()

# ⚔️ 物理 6-26 定位
l_idx = net.line[((net.line.from_bus == 6) & (net.line.to_bus == 26)) | 
                 ((net.line.from_bus == 26) & (net.line.to_bus == 6))].index
if len(l_idx) == 0:
    l_idx = net.line[((net.line.from_bus == 5) & (net.line.to_bus == 25)) | 
                     ((net.line.from_bus == 25) & (net.line.to_bus == 5))].index

net.line.loc[l_idx, 'in_service'] = False
print(f"✅ 成功斩断物理 6-26 线路 (Index: {l_idx.values[0]})")

pp.runpp(net, algorithm='bfsw')
Y_dense = net._ppc['internal']['Ybus'].toarray() if hasattr(net._ppc['internal']['Ybus'], 'toarray') else net._ppc['internal']['Ybus']
np.save('G_33_C3.npy', Y_dense.real)
np.save('B_33_C3.npy', Y_dense.imag)

dataset = []
for _ in tqdm(range(1000), desc="Generating C3"):
    noise = np.random.uniform(0.8, 1.2, size=len(net.load))
    net.load.p_mw, net.load.q_mvar = base_p * noise, base_q * noise
    try:
        pp.runpp(net, algorithm='bfsw')
        row = [net.res_bus.at[b, f] for b in range(33) for f in ['p_mw','q_mvar','vm_pu','va_degree']]
        dataset.append(row)
    except: continue
pd.DataFrame(dataset, columns=headers).to_csv("data_33_C3.csv", index=False)

✅ 成功斩断物理 6-26 线路 (Index: 24)


Generating C3: 100%|██████████| 1000/1000 [00:18<00:00, 53.23it/s]
